# Business Issue: 채권 RA (Risk Assessment)

**문제**: 신규 주문을 받을 때 현재 채권 잔액, 여신한도, 연체 aging을 함께 확인해야 합니다. 주문만 보면 매출 기회가 보이지만, 회수 리스크를 무시하면 한도 초과와 연체가 커질 수 있습니다.

이 실습에서 RA는 **Risk Assessment**입니다. Harness는 업무 요청을 Business Agent로 보내 근거를 패킷화하고, 여신 판단 수치는 코드로 재현 가능하게 계산합니다.

In [ ]:
# 합성 채권 snapshot으로 여신 정책을 투명하게 계산합니다.
case = {"request": "신규 주문의 신용 리스크를 평가해줘.", "credit": {"credit_limit_krw": 80_000_000, "current_exposure_krw": 75_000_000, "requested_order_krw": 18_000_000}, "aging_krw": {"current": 53_000_000, "days_1_30": 10_000_000, "days_31_60": 8_000_000, "days_61_90": 4_000_000, "over_90": 0}}
def assess_receivables(data):
    aging, credit = data["aging_krw"], data["credit"]; overdue = sum(v for k, v in aging.items() if k != "current"); post = credit["current_exposure_krw"] + credit["requested_order_krw"]; excess = max(0, post - credit["credit_limit_krw"])
    flags = ["credit_limit_exceeded"] if excess else []; flags += ["overdue_ratio_exceeded"] if overdue / sum(aging.values()) > .20 else []
    return {"overdue_krw": overdue, "overdue_ratio": overdue / sum(aging.values()), "post_order_exposure_krw": post, "over_limit_amount_krw": excess, "decision": "conditional_release" if flags else "eligible_for_credit_approval", "risk_flags": flags, "shipment_release_allowed": False, "action": "credit_manager_approval_required", "write_performed": False}


## 1. 채권·주문·신용정책 근거

실제 환경에서는 AR 원장, Sales Order, 신용정책에서 읽기 전용으로 가져옵니다. 여기서는 합성 snapshot만 사용합니다.

In [ ]:
print(case["request"])
for key, value in case["credit"].items(): print(f"- {key}: {value:,}원")
for bucket, value in case["aging_krw"].items(): print(f"  {bucket}: {value:,}원")


## 2. Harness → 채권 RA 정책 게이트

정책은 90일 초과 연체가 있으면 출고 보류하고, 한도 초과 또는 표준 연체비율 초과면 조건부 출고로 분류합니다. 이는 추천일 뿐이며 신용관리자가 해제하기 전에는 출고를 변경하지 않습니다.

In [ ]:
result = assess_receivables(case)
print("선택 specialist: business-agent (채권·주문 분석)")
for key in ["overdue_krw", "post_order_exposure_krw", "over_limit_amount_krw", "decision", "risk_flags"]: print(key, ":", result[key])
print("연체 비율:", f"{result['overdue_ratio']:.1%}")


## 3. Human-in-the-loop

Harness의 권고는 출고·한도 변경을 직접 실행하지 않습니다. 신용관리자는 회수 약정, 초과분 결제·담보, 예외 승인 근거를 확인한 뒤에만 별도의 승인된 write tool을 사용할 수 있습니다.

In [ ]:
print("상태:", result["action"]); print("출고 해제 허용:", result["shipment_release_allowed"]); print("실제 write 수행:", result["write_performed"])
